# 01 — Bank distances

Reduces raw WOCU bank point clouds to a single `dist_m` value per scope region per survey year.

**Algorithm:**
1. Keep only `status == 'OK'` points.
2. Select the `N_POINTS` furthest OK points per region × year.
3. Take their mean distance from the centreline → `dist_m`.

**Input:** `01_raw/erosion/wocu_output_fase2_20260210.gpkg` — layer `punten_oever`

**Output:** `03_features/EXPERIMENT/dist_per_year.parquet` — `(location_id, year, dist_m, n_ok_pts)`

In [ ]:
import os, sys
from pathlib import Path

_cwd = Path.cwd()
for candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (candidate / 'src').exists():
        _backend = candidate
        break
else:
    _backend = _cwd

os.chdir(_backend)
sys.path.insert(0, str(_backend))
print('cwd:', os.getcwd())

In [ ]:
import geopandas as gpd
import pandas as pd

import src.paths as PATHS

# ── Experiment config ─────────────────────────────────────────────────────────
EXPERIMENT   = '20260314'
N_POINTS     = 3       # top-N furthest OK points per region × year

RAW_GPKG     = PATHS.DATA_DIR / '01_raw/erosion/wocu_output_fase2_20260210.gpkg'
FEATURES_DIR = PATHS.DATA_DIR / f'03_features/{EXPERIMENT}'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

OUT_PARQUET  = FEATURES_DIR / 'dist_per_year.parquet'

print(f'Experiment : {EXPERIMENT}')
print(f'Features → : {FEATURES_DIR}')

## 1. Load raw bank points

In [ ]:
print('Reading punten_oever ...')
pts = gpd.read_file(RAW_GPKG, layer='punten_oever')
print(f'  {len(pts):,} rows  CRS: {pts.crs}')
print(f'  columns: {list(pts.columns)}')
print(f'  status values: {pts["status"].value_counts().to_dict()}')
print(f'  dtm_date years: {sorted(pts["dtm_date"].unique())}')

## 2. Filter to OK points and select top-N per region × year

In [ ]:
ok = pts[pts['status'] == 'OK'].copy()
print(f'OK points: {len(ok):,}  ({len(ok)/len(pts)*100:.1f}% of total)')

# Rename dtm_date → year (integer)
ok = ok.rename(columns={'dtm_date': 'year'})
ok['year'] = ok['year'].astype(int)

# Select top-N furthest OK points per (location_id, year)
def top_n_mean_dist(group, n=N_POINTS):
    chosen = group.nlargest(n, 'dist')
    return pd.Series({
        'dist_m':    chosen['dist'].mean(),
        'n_ok_pts':  len(group),           # total OK pts in this region × year
        'n_selected': len(chosen),
    })

dist_per_year = (
    ok.groupby(['location_id', 'year'], group_keys=False)
    .apply(top_n_mean_dist, include_groups=False)
    .reset_index()
)

print(f'\ndist_per_year shape: {dist_per_year.shape}')
print(f'  unique locations : {dist_per_year["location_id"].nunique():,}')
print(f'  years covered    : {sorted(dist_per_year["year"].unique())}')
dist_per_year.head(6)

## 3. Sanity checks

In [ ]:
# Every dist_m must be positive
assert (dist_per_year['dist_m'] > 0).all(), 'Some dist_m values are non-positive!'

# No duplicate (location_id, year) pairs
dupes = dist_per_year.duplicated(['location_id', 'year']).sum()
assert dupes == 0, f'{dupes} duplicate (location_id, year) pairs!'

# Timestamps per location
ts_counts = dist_per_year.groupby('location_id')['year'].count()
print('Timestamps per location:')
print(ts_counts.value_counts().sort_index().to_string())
print(f'\nTotal: {len(dist_per_year):,} rows across {ts_counts.index.nunique():,} locations')
print('All checks passed.')

## 4. Save

In [ ]:
dist_per_year.to_parquet(OUT_PARQUET, index=False)
print(f'Saved → {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size / 1e3:.0f} KB)')
print(f'Shape : {dist_per_year.shape}')
print(dist_per_year.dtypes)